In [1]:
#parameter counting before coding-

#MODEL A
#layer1- 3*4+4=16
#layer2- 1*4+1=5  therefore total=21

#MODEL B
#layer1- 3*6+6=24
#layer2-6*6+6=42
#layer3-1*6+1=7 therefore total=73

#MODEL C
#layer1-3*8+8=32
#layer2-8*8+8=72
#layer3-72
#layer4-72
#layer5-8*1+1=9 therefore total=257

#MODEL D
#layer1- 3*8+8=32
#layer 2,3,4,5,6,7,8- 72*7=504
#layer9- 8*1+1=9 therefore total=545

In [2]:
import numpy as np
np.random.seed(42)

In [3]:
X = np.random.uniform(-2, 2, (400, 3))

y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
)

y = y.reshape(-1, 1)


In [4]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

In [5]:

def initialize(layers):
    params = {}
    for i in range(1, len(layers)):
        params[f"W{i}"] = np.random.uniform(-0.5, 0.5, (layers[i], layers[i-1]))
        params[f"b{i}"] = np.zeros((layers[i], 1))
    return params

In [6]:
def forward(X, params, activation):
    A = X.T
    cache = {"A0": A}
    L = len(params) // 2

    for i in range(1, L + 1):
        Z = params[f"W{i}"] @ A + params[f"b{i}"]
        cache[f"Z{i}"] = Z

        if i == L:
            A = Z  # Output layer is linear
        else:
            A = activation(Z)

        cache[f"A{i}"] = A

    return A, cache

In [7]:
def compute_loss(y_true, y_pred):
    return np.mean((y_true - y_pred.T) ** 2)

In [8]:
def backward(X, y, params, cache, activation_derivative):
    grads = {}
    N = X.shape[0]
    L = len(params) // 2

    y = y.T
    dA = cache[f"A{L}"] - y

    for i in reversed(range(1, L + 1)):

        if i == L:
            dZ = dA
        else:
            dZ = dA * activation_derivative(cache[f"Z{i}"])

        grads[f"dW{i}"] = (1 / N) * dZ @ cache[f"A{i-1}"].T
        grads[f"db{i}"] = (1 / N) * np.sum(dZ, axis=1, keepdims=True)

        if i > 1:
            dA = params[f"W{i}"].T @ dZ

    return grads

In [9]:
def update(params, grads, lr=0.01):
    L = len(params) // 2
    for i in range(1, L + 1):
        params[f"W{i}"] -= lr * grads[f"dW{i}"]
        params[f"b{i}"] -= lr * grads[f"db{i}"]
    return params

In [10]:
def gradient_norm(matrix):
    return np.sqrt(np.sum(matrix**2))

In [11]:
def train_model(layers, activation, activation_derivative, name):

    params = initialize(layers)
    losses = []
    epochs = 1000

    for epoch in range(epochs):
        y_pred, cache = forward(X, params, activation)
        loss = compute_loss(y, y_pred)
        losses.append(loss)

        grads = backward(X, y, params, cache, activation_derivative)
        params = update(params, grads, lr=0.01)

    L = len(params)//2
    grad_L1 = gradient_norm(grads["dW1"])
    grad_last_hidden = gradient_norm(grads[f"dW{L-1}"]) if L > 1 else grad_L1

    print("\n")
    print(name)
    print("Final Loss:", losses[-1])
    print("Loss at Epoch 200:", losses[199])
    print("Gradient Norm Layer 1:", grad_L1)
    print("Gradient Norm Last Hidden:", grad_last_hidden)

    return losses

In [12]:
layers_A = [3, 4, 1]
train_model(layers_A, relu, relu_derivative, "Model A - Shallow (ReLU)")



Model A - Shallow (ReLU)
Final Loss: 0.25213400497919336
Loss at Epoch 200: 0.691935381524139
Gradient Norm Layer 1: 0.07509076426690273
Gradient Norm Last Hidden: 0.07509076426690273


[np.float64(3.219058085818201),
 np.float64(3.161143423163921),
 np.float64(3.1056008442558185),
 np.float64(3.052318237658292),
 np.float64(3.001133785479145),
 np.float64(2.951887545805426),
 np.float64(2.904476785795903),
 np.float64(2.8587672728231333),
 np.float64(2.8146678496447204),
 np.float64(2.772082478735695),
 np.float64(2.7309201404923464),
 np.float64(2.6911022598798815),
 np.float64(2.6525466879955553),
 np.float64(2.615183112871333),
 np.float64(2.5789533805356917),
 np.float64(2.5438094066598897),
 np.float64(2.509669553220894),
 np.float64(2.476495555843716),
 np.float64(2.4442456535067767),
 np.float64(2.4128661801106426),
 np.float64(2.3823034196205053),
 np.float64(2.3525205726700116),
 np.float64(2.3234934627279227),
 np.float64(2.295183395283841),
 np.float64(2.267539249210235),
 np.float64(2.2405308806741293),
 np.float64(2.214131744509587),
 np.float64(2.1883141454635373),
 np.float64(2.1630504998949074),
 np.float64(2.1383160986742107),
 np.float64(2.114081592

In [13]:
layers_B = [3, 6, 6, 1]
train_model(layers_B, relu, relu_derivative, "Model B - Medium (ReLU)")



Model B - Medium (ReLU)
Final Loss: 0.11516575902400936
Loss at Epoch 200: 0.5944602379956442
Gradient Norm Layer 1: 0.03980593232437231
Gradient Norm Last Hidden: 0.023261373013213252


[np.float64(2.131636934532029),
 np.float64(2.1164457982247256),
 np.float64(2.101484188393568),
 np.float64(2.0867430791621713),
 np.float64(2.0722056098178485),
 np.float64(2.0579075618631784),
 np.float64(2.0438384580443887),
 np.float64(2.0299888495956764),
 np.float64(2.016329616552811),
 np.float64(2.0028460006996975),
 np.float64(1.9895346487973893),
 np.float64(1.9764007610370828),
 np.float64(1.9634265479251491),
 np.float64(1.950639715284484),
 np.float64(1.9380107684356813),
 np.float64(1.9255709096297233),
 np.float64(1.9133119693539158),
 np.float64(1.9012228342080897),
 np.float64(1.8893000037930836),
 np.float64(1.8775372547717046),
 np.float64(1.865918497848193),
 np.float64(1.8544245143707354),
 np.float64(1.8430602156296991),
 np.float64(1.8318338442552295),
 np.float64(1.8207364816724896),
 np.float64(1.8097250068402886),
 np.float64(1.7988537443025467),
 np.float64(1.7881088989514171),
 np.float64(1.777487987306666),
 np.float64(1.7669605002524034),
 np.float64(1.75

In [14]:
layers_C = [3, 8, 8, 8, 8, 1]
train_model(layers_C, relu, relu_derivative, "Model C - Deep (ReLU)")



Model C - Deep (ReLU)
Final Loss: 0.06326060171937394
Loss at Epoch 200: 1.647754271366167
Gradient Norm Layer 1: 0.031101023938903072
Gradient Norm Last Hidden: 0.020044598493259888


[np.float64(2.4414467486607645),
 np.float64(2.4255334241607533),
 np.float64(2.409934660211569),
 np.float64(2.3946777571340236),
 np.float64(2.379736676530582),
 np.float64(2.3651358963697677),
 np.float64(2.3508606991913323),
 np.float64(2.336914500204119),
 np.float64(2.3232947943494215),
 np.float64(2.309996598728895),
 np.float64(2.297024987487263),
 np.float64(2.284376060570258),
 np.float64(2.272057443783213),
 np.float64(2.2600557143064877),
 np.float64(2.248320832243853),
 np.float64(2.23684513433168),
 np.float64(2.2256345567102525),
 np.float64(2.2147014432520495),
 np.float64(2.2040088319847313),
 np.float64(2.193549698560403),
 np.float64(2.1833195909957213),
 np.float64(2.1733128913910464),
 np.float64(2.163518399744989),
 np.float64(2.1539285106518293),
 np.float64(2.144536635715565),
 np.float64(2.1353391616363755),
 np.float64(2.126335090044272),
 np.float64(2.117516042508679),
 np.float64(2.10887700044021),
 np.float64(2.100417151343408),
 np.float64(2.09213230822985

In [15]:
layers_D = [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]
train_model(layers_D, relu, relu_derivative, "Model D - Very Deep (ReLU)")



Model D - Very Deep (ReLU)
Final Loss: 0.11072966920787074
Loss at Epoch 200: 1.723374673177218
Gradient Norm Layer 1: 0.07402017533952712
Gradient Norm Last Hidden: 0.0351744730147196


[np.float64(2.45080795367158),
 np.float64(2.4314292610273065),
 np.float64(2.411985305720724),
 np.float64(2.3931571013965196),
 np.float64(2.3749150463210054),
 np.float64(2.3571969698215356),
 np.float64(2.339917384516383),
 np.float64(2.322982330242246),
 np.float64(2.3062041422533492),
 np.float64(2.28968485827741),
 np.float64(2.2735616613118994),
 np.float64(2.2578814538546728),
 np.float64(2.242617774379044),
 np.float64(2.2277871124300854),
 np.float64(2.213387205293358),
 np.float64(2.1993575140690784),
 np.float64(2.1857610475976053),
 np.float64(2.172532762504519),
 np.float64(2.1596749111976523),
 np.float64(2.1471359532514485),
 np.float64(2.1350124241105597),
 np.float64(2.1232208671165744),
 np.float64(2.111673056051592),
 np.float64(2.1005235210175806),
 np.float64(2.0895972462128403),
 np.float64(2.0790748761163043),
 np.float64(2.0687581732542197),
 np.float64(2.05884087257852),
 np.float64(2.0490822119903607),
 np.float64(2.039654005763752),
 np.float64(2.0304676616

In [16]:
#OBSERVATION TABLE-
import pandas as pd
data = {
    "Model": ["Model A", "Model B", "Model C", "Model D", "Model D"],
    "Activation": ["ReLU", "ReLU", "ReLU", "ReLU", "Sigmoid"],
    "Final Loss": [0.2521, 0.1152, 0.0633, 0.1107, 1.7439],
    "Loss @ Epoch 200": [0.6919, 0.5945, 1.6478, 1.7234, 1.7439],
    "Grad Norm (Layer 1)": [0.0751, 0.0398, 0.0311, 0.0740, 3.15e-06],
    "Grad Norm (Last Hidden)": [0.0751, 0.0233, 0.0200, 0.0352, 3.17e-06],
    "Training Behavior": [
        "Stable, smooth convergence",
        "Stable, improved performance",
        "Slow start, best final fit",
        "Stable but slower, no improvement over C",
        "Extremely slow, early stagnation"
    ]
}

df = pd.DataFrame(data)
df

,Model,Activation,Final Loss,Loss @ Epoch 200,Grad Norm (Layer 1),Grad Norm (Last Hidden),Training Behavior
0,Model A,ReLU,0.2521,0.6919,0.075100,0.075100,"Stable, smooth convergence"
1,Model B,ReLU,0.1152,0.5945,0.039800,0.023300,"Stable, improved performance"
2,Model C,ReLU,0.0633,1.6478,0.031100,0.020000,"Slow start, best final fit"
3,Model D,ReLU,0.1107,1.7234,0.074000,0.035200,"Stable but slower, no improvement over C"
4,Model D,Sigmoid,1.7439,1.7439,0.000003,0.000003,"Extremely slow, early stagnation"


In [17]:
#reflection questions

#1.
'''
Model A → Model B → Model C showed improved final lossHowever, Model D (deeper than Model C) did not improve further.Model D (ReLU) had worse final loss than Model C.
Model D (Sigmoid) performed very poorly and stagnated.Also, deeper models (C and D) learned slower in early epochs compared to shallow models.
Therefore, increasing depth did not always reduce loss faster or guarantee better performance.
'''

#2.
'''
In Model A (shallow), gradient norms were equal (only one hidden layer).In Model B and C, gradient norms decreased as we moved toward earlier layers.
In Model D (Sigmoid), gradients became extremely small (~10⁻⁶) in all layers.This shows that gradient magnitudes shrink as depth increases, especially with Sigmoid activation.'''

#3.
'''
No.ReLU models trained stably across all depths.Sigmoid in the very deep network caused extremely slow learning.Model D (Sigmoid) plateaued early and barely improved after 200 epochs.'''

#4.
'''
ReLU behaved significantly more stable than Sigmoid.Evidence from observations:ReLU gradient norms remained in a reasonable range (0.02–0.07).Sigmoid gradient norms dropped to ~3 × 10⁻⁶.'''

#5.
'''
Yes.All models used the same learning rate (0.01), but Model C and Model D (ReLU) showed slow initial improvement.'''

'\nYes.All models used the same learning rate (0.01), but Model C and Model D (ReLU) showed slow initial improvement.'